###Delta Table – CTAS (Create Table As Select) Example
- Overview
- CTAS (Create Table As Select) is a Databricks SQL command used to:
- 
- Create a new Delta table
- Populate it immediately with data from a SELECT query
####Advantages of CTAS:

- Create & populate table in one step
- Avoids separate CREATE TABLE + INSERT INTO
- Supports transformations during table creation
- Can specify table properties, partitioning, clustering, and storage location
####Syntax:

> CREATE TABLE <table_name>
> USING <format>
> [LOCATION <path>]
> AS
> SELECT ...;
#####Advantages:
- Creates & populates table in one step.
- Supports transformations (casting, filtering, computed columns) during creation.
- Supports Delta table properties, partitioning, and clustering.
- Reduces need for separate CREATE TABLE + INSERT operations.

In [0]:

# Step 1: Prepare sample transaction data as DataFrame
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType

spark = SparkSession.builder.getOrCreate()

data = [
  ("00000000","06-26-2011","4007024", 40.33,"Exercise & Fitness","Cardio Machine Accessories","Clarksville","Tennessee","credit"),
  ("00000001","05-26-2011","4006742", 198.44,"Exercise & Fitness","Weightlifting Gloves","Long Beach","California","credit"),
  ("00000002","06-01-2011","4009775", 5.58,"Exercise & Fitness","Weightlifting Machine Accessories","Anaheim","California","credit"),
  ("00000003","06-05-2011","4002199", 198.19,"Gymnastics","Gymnastics Rings","Milwaukee","Wisconsin","credit"),
  ("00000004","12-17-2011","4002613", 98.81,"Team Sports","Field Hockey","Nashville","Tennessee","credit"),
  ("00000005","02-14-2011","4007591", 193.63,"Outdoor Recreation","Camping & Backpacking & Hiking","Chicago","Illinois","credit"),
  ("00000006","10-28-2011","4002190", 27.89,"Puzzles","Jigsaw Puzzles","Charleston","South Carolina","credit"),
  ("00000007","07-14-2011","4002964", 96.01,"Outdoor Play Equipment","Sandboxes","Columbus","Ohio","credit"),
  ("00000008","01-17-2011","4007361", 10.44,"Winter Sports","Snowmobiling","Des Moines","Iowa","credit")
]

schema = StructType([
    StructField("txnid", StringType(), True),
    StructField("txndate", StringType(), True),   # keep as string for now, or parse to DateType
    StructField("custid", StringType(), True),
    StructField("amount", DoubleType(), True),    # numeric type for calculations
    StructField("product", StringType(), True),
    StructField("category", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("paytype", StringType(), True)
])
df_txn = spark.createDataFrame(data, schema)

display(df_txn)
     

In [0]:

## Step 2: Write the DataFrame to a staging Delta table
# We will create a **temporary table** to use for the CTAS example.

df_txn.write.format("delta").mode("overwrite").saveAsTable("new_catalog.default_schema.txn_staging")
print("Staging Table Created")
     

###Step 3: Create a new table using CTAS
- The new table txn_ctas is created and populated in a single command.
- We can also apply transformations during creation (e.g., cast amount to double, convert date).

In [0]:
%sql
-- Step 3: Create curated Delta table using CTAS
-- This builds txn_ctas from the staging table.
-- Key transformations:
--   - Convert txndate string to DATE type
--   - Cast amount from string to DOUBLE
--   - Preserve other fields as-is

CREATE TABLE new_catalog.default_schema.txn_ctas
USING DELTA
AS
SELECT
  txnid,
  to_date(txndate,'MM-dd-yyyy') AS txndate, -- Parse string into DATE
  custid,
  CAST(amount AS DOUBLE) AS amount,         -- Convert to numeric type
  product,
  category,
  city,
  state,
  paytype
FROM new_catalog.default_schema.txn_staging;

In [0]:
%sql
-- Step 4: Query the new CTAS table
-- This confirms that the curated table was created successfully
-- and displays the transformed records.

SELECT * 
FROM new_catalog.default_schema.txn_ctas;

-- Inspect table metadata
-- DESCRIBE FORMATTED shows schema, provider, location, and properties.
-- Useful for auditing and verifying Delta table configuration.

DESCRIBE FORMATTED new_catalog.default_schema.txn_ctas;

In [0]:
%sql
-- Step 5: Create a specialized CTAS table
-- This builds txn_ctas_exercise from txn_ctas.
-- Only transactions where category = 'Exercise & Fitness' are included.

CREATE TABLE new_catalog.default_schema.txn_ctas_exercise
USING DELTA
AS
SELECT *
FROM new_catalog.default_schema.txn_ctas
WHERE category = 'Exercise & Fitness';

In [0]:
%sql
-- Step 6: Create optimized CTAS table
-- This builds txn_ctas_opt from the staging table.
-- Key features:
--   - Partitioned by state (efficient filtering)
--   - Change Data Capture enabled (track row-level changes)
--   - Auto Optimize enabled (optimize write and compact small files)

CREATE TABLE new_catalog.default_schema.txn_ctas_opt
USING DELTA
PARTITIONED BY (state)
TBLPROPERTIES (
  'delta.enableChangeDataCapture' = 'true',        -- CDC for auditing and downstream pipelines
  'delta.autoOptimize.optimizeWrite' = 'true',     -- Optimize file sizes during writes
  'delta.autoOptimize.autoCompact' = 'true'        -- Compact small files automatically
)
AS
SELECT
  txnid,
  to_date(txndate,'MM-dd-yyyy') AS txndate,  -- Convert to DATE type
  custid,
  CAST(amount AS DOUBLE) AS amount,          -- Numeric type for calculations
  product,
  category,
  city,
  state,
  paytype
FROM new_catalog.default_schema.txn_staging;